# ⚡ Hardware Roofline Profiling & Triton Kernel Optimization with NeuralOps

In this notebook, we analyze arithmetic intensity, peak SRAM bandwidth savings, and compiled OpenAI Triton scan speedups using NeuralOps' physical Hardware Roofline Analyzer.

In [ ]:
!pip install neuralops torch triton
import torch
import neuralops
from neuralops.core.roofline import profile_roofline, render_roofline_terminal

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Run Hardware Roofline Profiler
Profile FLOPs, HBM bytes fused in SRAM, and Triton vs PyTorch latency.

In [ ]:
report = profile_roofline(
    module_name="gated_delta_net",
    batch_size=2,
    seq_len=2048,
    embed_dim=768,
    num_heads=12,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    warmup=5,
    rep=10,
)

print(f"Total FLOPs: {report.total_flops / 1e9:.2f} GFLOPs")
print(f"Fused SRAM Traffic: {report.sram_bytes_saved_mb:.2f} MB")
print(f"Triton Latency: {report.latency_triton_ms:.3f} ms")
print(f"PyTorch Fallback: {report.latency_pytorch_ms:.3f} ms")
print(f"Measured Speedup: {report.speedup_x:.2f}x")

## 2. Render ASCII Roofline Chart

In [ ]:
chart = render_roofline_terminal(report)
print(chart)